# E8: Feature Selection Funnel — L4 (Redundancy Removal)

**Spec section**: 6.5  
**Input**: DEV GOOD+BAD; features with `L3_pass = Y`; WoE from L1 binning objects.  
**Output**: `l4_pool.json`, correlation matrix, VIF table, source diversity report.

In [0]:
%pip install "optbinning" "numpy<2" "pyarrow==14.0.1" --quiet
dbutils.library.restartPython()

In [0]:
import sys, os, hashlib, shutil, tempfile, json, time, pickle, warnings
from datetime import datetime

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from statsmodels.stats.outliers_influence import variance_inflation_factor
import common

warnings.filterwarnings("ignore", category=FutureWarning)

cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
ARTIFACTS = common.artifacts_path(cfg)
REPORTS   = os.path.join(ARTIFACTS, "reports")
BINNING_DIR = os.path.join(ARTIFACTS, "binning", "L1")
FEAT_LISTS  = os.path.join(ARTIFACTS, "feature_lists")
os.makedirs(REPORTS, exist_ok=True)
os.makedirs(FEAT_LISTS, exist_ok=True)

cat = cfg["databricks"]["catalog"]
sch = cfg["databricks"]["schema"]
MASTER_TBL = f"{cat}.{sch}.acard_master"
FLOG_TBL   = f"{cat}.{sch}.acard_feature_log"

L4 = cfg["funnel"]["L4"]
CORR_MAX    = L4["corr_max"]
CORR_METHOD = L4.get("corr_method", "spearman")
VIF_MAX     = L4["vif_max"]

target_col = cfg["columns"]["target"]
status_col = cfg["columns"]["status"]
SEED = cfg["random_seed"]

# Funnel config hash check
funnel_config_hash = hashlib.sha256(
    json.dumps(cfg["funnel"], sort_keys=True).encode()
).hexdigest()

run_summary_path = os.path.join(REPORTS, "run_summaries", "s03_L0.json")
with open(run_summary_path) as f:
    l0_summary = json.load(f)
l0_funnel_hash = l0_summary["funnel_config_hash"]

tc_path = os.path.join(PROJECT_ROOT, "config", "threshold_changes.csv")
if funnel_config_hash != l0_funnel_hash:
    if os.path.exists(tc_path):
        tc = pd.read_csv(tc_path)
        print(f"\u26a0 Funnel config hash changed but threshold_changes.csv has {len(tc)} row(s) — proceeding.")
    else:
        raise ValueError(
            f"Funnel config hash mismatch!\n"
            f"  L0 frozen : {l0_funnel_hash[:12]}\n"
            f"  Current   : {funnel_config_hash[:12]}"
        )
else:
    print(f"\u2705 Funnel config hash matches L0: {funnel_config_hash[:12]}")

_TMP = tempfile.mkdtemp(prefix="e8_")
_t0 = time.time()

print(f"Config hash : {common.config_hash(cfg)[:12]}")
print(f"L4 params   : corr_max={CORR_MAX}, corr_method={CORR_METHOD}, vif_max={VIF_MAX}")

In [0]:
# ── Load master, feature log, binning objects ──────────────────
master = spark.table(MASTER_TBL).toPandas()
dev_gb = master[
    (master["window"] == "DEV") & (master[status_col].isin(cfg["labelled_statuses"]))
].copy()
y = (dev_gb[status_col] == "BAD").astype(int)
print(f"DEV GOOD+BAD: {len(dev_gb):,}  Bads: {y.sum()}")

flog = spark.table(FLOG_TBL).toPandas()
l3_pass = flog[flog["L3_pass"] == "Y"].copy()
l3_feats = l3_pass["feature"].tolist()
l3_feats = [f for f in l3_feats if f in dev_gb.columns]
print(f"L3 survivors: {len(l3_feats)}")

# Load L1 binning objects
binning_objects = {}
for feat in l3_feats:
    with open(os.path.join(BINNING_DIR, f"{feat}.pkl"), "rb") as f:
        binning_objects[feat] = pickle.load(f)

# Build WoE matrix using L1 bins (transform metric="woe")
def safe_woe(ob, X):
    """WoE transform with numeric coercion fallback."""
    try:
        return ob.transform(X, metric="woe")
    except TypeError:
        return ob.transform(pd.to_numeric(X, errors="coerce"), metric="woe")

print("Building WoE matrix...")
woe_cols = {}
for feat in l3_feats:
    woe_cols[feat] = safe_woe(binning_objects[feat], dev_gb[feat])
X_woe = pd.DataFrame(woe_cols, index=dev_gb.index)

# Replace infinities with NaN, then fill NaN with 0 (neutral WoE)
X_woe.replace([np.inf, -np.inf], np.nan, inplace=True)
X_woe.fillna(0.0, inplace=True)

print(f"WoE matrix: {X_woe.shape}")

# Feature dictionary for special_codes, trend
fdict = pd.read_csv(os.path.join(PROJECT_ROOT, "config", "feature_dict.csv"))
fdict_idx = fdict.set_index("feature")

In [0]:
# ── Priority ranking (spec 6.5.1) ─────────────────────────────
# 1. psi_status: green=0, yellow=1, red-kept=2
# 2. L2_flag: N=0, Y=1
# 3. iv_dev descending

ranking = l3_pass[["feature", "psi_status", "L2_flag", "iv_dev"]].copy()
ranking = ranking[ranking["feature"].isin(l3_feats)].copy()

ranking["psi_rank"] = ranking["psi_status"].map({"green": 0, "yellow": 1, "red": 2}).fillna(2).astype(int)
ranking["flag_rank"] = ranking["L2_flag"].map({"N": 0, "Y": 1}).fillna(1).astype(int)
ranking = ranking.sort_values(
    ["psi_rank", "flag_rank", "iv_dev"],
    ascending=[True, True, False]
).reset_index(drop=True)
ranking["priority"] = range(1, len(ranking) + 1)

print(f"Priority ranking ({len(ranking)} features):")
print(f"  Top-5: {ranking['feature'].head().tolist()}")
print(f"  Bottom-5: {ranking['feature'].tail().tolist()}")
print(f"\n  By PSI status: {ranking.groupby('psi_rank').size().to_dict()}")
print(f"  By L2 flag  : {ranking.groupby('flag_rank').size().to_dict()}")

ordered_feats = ranking["feature"].tolist()

In [0]:
# ── Greedy correlation de-duplication (spec 6.5.2) ─────────────

print(f"Running greedy correlation filter (|r| <= {CORR_MAX}, {CORR_METHOD})...")

# Precompute full correlation matrix
corr_matrix = X_woe[ordered_feats].corr(method=CORR_METHOD)

kept = []
dropped_corr = []  # (feature, corr_with, abs_r)

for feat in ordered_feats:
    if not kept:
        kept.append(feat)
        continue
    # Check correlation against all already-kept features
    corrs = corr_matrix.loc[feat, kept].abs()
    max_r = corrs.max()
    if max_r <= CORR_MAX:
        kept.append(feat)
    else:
        worst_with = corrs.idxmax()
        dropped_corr.append((feat, worst_with, float(max_r)))

print(f"After correlation filter: {len(kept)} kept, {len(dropped_corr)} dropped")
if dropped_corr:
    print(f"  First 5 drops:")
    for f, w, r in dropped_corr[:5]:
        print(f"    {f} -> corr_with={w} |r|={r:.3f}")

In [0]:
# ── VIF iterative removal (spec 6.5.3) ───────────────────────

print(f"Running VIF filter (max VIF <= {VIF_MAX})...")

# Build priority lookup for tie-breaking
priority_map = {f: i for i, f in enumerate(ordered_feats)}

vif_pool = list(kept)  # start from correlation-filtered pool
dropped_vif = []

def compute_vif(features):
    """Compute VIF for all features (WoE + constant)."""
    X = X_woe[features].values
    X_const = np.column_stack([np.ones(X.shape[0]), X])
    vifs = {}
    for i, feat in enumerate(features):
        vif_val = variance_inflation_factor(X_const, i + 1)  # +1 because col 0 is const
        vifs[feat] = vif_val
    return vifs

max_iters = 200
for iteration in range(max_iters):
    if len(vif_pool) <= 1:
        break
    vifs = compute_vif(vif_pool)
    max_vif_feat = max(vifs, key=vifs.get)
    max_vif_val = vifs[max_vif_feat]
    if max_vif_val <= VIF_MAX:
        break
    # Tie-break: if multiple features have the same max VIF, drop the lowest priority
    max_vif_feats = [f for f, v in vifs.items() if abs(v - max_vif_val) < 1e-6]
    if len(max_vif_feats) > 1:
        max_vif_feat = max(max_vif_feats, key=lambda f: priority_map.get(f, 999))
    dropped_vif.append((max_vif_feat, max_vif_val))
    vif_pool.remove(max_vif_feat)
    if (iteration + 1) % 5 == 0:
        print(f"  iter {iteration+1}: dropped {max_vif_feat} (VIF={max_vif_val:.1f}), pool={len(vif_pool)}")

# Final VIF check
final_vifs = compute_vif(vif_pool) if len(vif_pool) > 1 else {vif_pool[0]: 1.0}
print(f"\nAfter VIF filter: {len(vif_pool)} kept, {len(dropped_vif)} dropped")
print(f"Max final VIF: {max(final_vifs.values()):.2f}")
if dropped_vif:
    print(f"  First 5 VIF drops:")
    for f, v in dropped_vif[:5]:
        print(f"    {f} (VIF={v:.1f})")

In [0]:
# ── Source diversity (spec 6.5.4) ─────────────────────────────

pool_set = set(vif_pool)

# Source counts in the final pool
pool_sources = l3_pass[l3_pass["feature"].isin(pool_set)].groupby("source").size()
print("Kept features per source:")
print(pool_sources.to_string())

# Check sources that had L3-surviving features with iv_dev >= 0.05 but none kept
l3_iv05 = l3_pass[l3_pass["iv_dev"] >= 0.05]
sources_with_good_feats = set(l3_iv05["source"].unique())
sources_with_kept = set(l3_pass[l3_pass["feature"].isin(pool_set)]["source"].unique())
missing_sources = sources_with_good_feats - sources_with_kept

print(f"\nSources with iv>=0.05 at L3 but no features kept: {missing_sources if missing_sources else 'none'}")

source_diversity_report = []
for src in missing_sources:
    candidates = l3_pass[(l3_pass["source"] == src) & (l3_pass["iv_dev"] >= 0.05)]
    candidates = candidates.sort_values("iv_dev", ascending=False)
    for _, row in candidates.head(5).iterrows():
        source_diversity_report.append({
            "source": src, "feature": row["feature"],
            "iv_dev": row["iv_dev"], "psi_status": row["psi_status"]
        })
        print(f"  Candidate: {row['feature']} (iv={row['iv_dev']:.4f}, psi={row['psi_status']})")

if not missing_sources:
    print("All sources with strong features are represented.")

In [0]:
# ── Save l4_pool.json and update feature log ──────────────────

# Build pool list (ordered by priority)
pool_ordered = [f for f in ordered_feats if f in pool_set]
pool_records = []
for feat in pool_ordered:
    rec = {"feature": feat}
    if feat in fdict_idx.index:
        rec["source"] = fdict_idx.loc[feat, "source"] if "source" in fdict_idx.columns else ""
        rec["dtype"] = fdict_idx.loc[feat, "dtype"] if "dtype" in fdict_idx.columns else ""
        rec["trend"] = flog.loc[flog["feature"] == feat, "trend"].values[0] if "trend" in flog.columns else ""
        rec["special_codes"] = fdict_idx.loc[feat, "special_codes"] if "special_codes" in fdict_idx.columns else ""
    pool_records.append(rec)

pool_path = os.path.join(FEAT_LISTS, "l4_pool.json")
with open(pool_path, "w") as f:
    json.dump(pool_records, f, indent=2, default=str)
print(f"Saved pool: {pool_path} ({len(pool_records)} features)")

# Build L4 results DataFrame
l4_results = []
for feat in l3_feats:
    row = {"feature": feat}
    if feat in pool_set:
        row["in_pool_BC"] = "Y"
        row["L4_pass"] = "Y"
        row["L4_reason"] = pd.NA
        row["vif"] = final_vifs.get(feat, np.nan)
    else:
        row["in_pool_BC"] = "N"
        row["L4_pass"] = "N"
        # Determine reason
        corr_drop = [d for d in dropped_corr if d[0] == feat]
        vif_drop = [d for d in dropped_vif if d[0] == feat]
        if corr_drop:
            row["L4_reason"] = f"CORR:{corr_drop[0][1]}"
            row["corr_with"] = corr_drop[0][1]
            row["corr_r"] = corr_drop[0][2]
        elif vif_drop:
            row["L4_reason"] = f"VIF:{vif_drop[0][1]:.1f}"
        else:
            row["L4_reason"] = "UNKNOWN"
        row["vif"] = np.nan
    l4_results.append(row)

l4_df = pd.DataFrame(l4_results)

# Merge into feature log
l4_merge_cols = ["feature", "in_pool_BC", "L4_pass", "L4_reason", "vif"]
if "corr_with" in l4_df.columns:
    l4_merge_cols.extend(["corr_with", "corr_r"])
l4_merge = l4_df[[c for c in l4_merge_cols if c in l4_df.columns]].copy()

l4_only = [c for c in l4_merge.columns if c != "feature"]
flog_clean = flog.drop(columns=[c for c in l4_only if c in flog.columns])
flog_updated = flog_clean.merge(l4_merge, on="feature", how="left")

mask_l4_drop = (flog_updated["L4_pass"] == "N") & (flog_updated["dropped_at"].isna())
flog_updated.loc[mask_l4_drop, "dropped_at"] = "L4"
flog_updated.loc[mask_l4_drop, "drop_rule"] = flog_updated.loc[mask_l4_drop, "L4_reason"]

for c in flog_updated.columns:
    if flog_updated[c].isna().all():
        flog_updated[c] = flog_updated[c].astype(str)

sdf = spark.createDataFrame(flog_updated)
sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(FLOG_TBL)
print(f"Saved table: {FLOG_TBL}  ({spark.table(FLOG_TBL).count()} rows)")

csv_path = os.path.join(ARTIFACTS, "feature_log.csv")
flog_updated.to_csv(csv_path, index=False)
print(f"Saved CSV : {csv_path}")

# ── Report: 03_L4.xlsx ──────────────────────────────────────
tmp_xl = os.path.join(_TMP, "03_L4.xlsx")

# Correlation matrix of kept features
corr_kept = X_woe[pool_ordered].corr(method=CORR_METHOD)

# VIF table
vif_table = pd.DataFrame([
    {"feature": f, "vif": final_vifs[f], "source": fdict_idx.loc[f, "source"] if f in fdict_idx.index else "",
     "iv_dev": float(l3_pass.loc[l3_pass["feature"] == f, "iv_dev"].values[0]) if len(l3_pass[l3_pass["feature"] == f]) > 0 else np.nan}
    for f in pool_ordered
]).sort_values("vif", ascending=False)

with pd.ExcelWriter(tmp_xl, engine="openpyxl") as writer:
    meta = pd.DataFrame([{
        "module": "E8_L4", "run_time": datetime.now().isoformat(),
        "config_hash": common.config_hash(cfg),
        "funnel_config_hash": funnel_config_hash,
        "features_in": len(l3_feats), "features_after_corr": len(kept),
        "features_after_vif": len(vif_pool), "corr_max": CORR_MAX, "vif_max": VIF_MAX,
    }])
    meta.to_excel(writer, sheet_name="meta", index=False)

    l4_df.sort_values(["L4_pass", "L4_reason"], ascending=[False, True]).to_excel(
        writer, sheet_name="summary", index=False
    )

    corr_kept.to_excel(writer, sheet_name="corr_matrix")
    vif_table.to_excel(writer, sheet_name="vif_table", index=False)

    if source_diversity_report:
        pd.DataFrame(source_diversity_report).to_excel(
            writer, sheet_name="source_diversity", index=False
        )
    else:
        pd.DataFrame([{"note": "All sources represented"}]).to_excel(
            writer, sheet_name="source_diversity", index=False
        )

    # Correlation drop detail
    if dropped_corr:
        pd.DataFrame(dropped_corr, columns=["feature", "corr_with", "abs_r"]).to_excel(
            writer, sheet_name="corr_drops", index=False
        )

shutil.copyfile(tmp_xl, os.path.join(REPORTS, "03_L4.xlsx"))
print(f"Saved report: {REPORTS}/03_L4.xlsx")

In [0]:
# ── Acceptance Criteria (E8 — L4) ────────────────────────────────
print("=" * 60)
print("ACCEPTANCE CRITERIA (E8 — L4)")
print("=" * 60)
checks = []

# 1. Pairwise |Spearman| <= corr_max among kept features
corr_pool = X_woe[pool_ordered].corr(method=CORR_METHOD)
corr_pool_chk = corr_pool.copy()
for i in range(len(corr_pool_chk)):
    corr_pool_chk.iat[i, i] = 0.0
max_abs_corr = corr_pool_chk.abs().values.max()
checks.append((f"Pairwise |Spearman| <= {CORR_MAX} (max={max_abs_corr:.4f})", max_abs_corr <= CORR_MAX + 1e-9))

# Find the worst pair for reporting
if max_abs_corr > 0:
    idx_flat = np.argmax(corr_pool_chk.abs().values)
    i, j = divmod(idx_flat, len(pool_ordered))
    print(f"  Worst pair: {pool_ordered[i]} vs {pool_ordered[j]} = {corr_pool_chk.values[i,j]:.4f}")

# 2. All VIF <= vif_max
max_vif = max(final_vifs.values())
checks.append((f"All VIF <= {VIF_MAX} (max={max_vif:.2f})", max_vif <= VIF_MAX + 1e-9))

# 3. Pool size reported
print(f"  Pool size: {len(vif_pool)}")
checks.append((f"Pool size reported ({len(vif_pool)} features)", True))

# 4. feature_log.csv exported
checks.append(("feature_log.csv exported", os.path.exists(csv_path)))

# 5. l4_pool.json saved
checks.append(("l4_pool.json saved", os.path.exists(pool_path)))

# 6. Feature log table rows
tbl_ct = spark.table(FLOG_TBL).count()
checks.append((f"Feature log table rows = {tbl_ct}", tbl_ct == len(flog_updated)))

# 7. Report saved
checks.append(("03_L4.xlsx report saved", os.path.exists(os.path.join(REPORTS, "03_L4.xlsx"))))

all_pass = True
for desc, ok in checks:
    status = "\u2705 PASS" if ok else "\u274c FAIL"
    print(f"  {status} : {desc}")
    if not ok:
        all_pass = False

# Run summary
summary = {
    "module": "E8_L4",
    "run_time": datetime.now().isoformat(),
    "elapsed_s": round(time.time() - _t0, 1),
    "config_hash": common.config_hash(cfg),
    "funnel_config_hash": funnel_config_hash,
    "features_in": len(l3_feats),
    "features_after_corr": len(kept),
    "features_after_vif": len(vif_pool),
    "corr_dropped": len(dropped_corr),
    "vif_dropped": len(dropped_vif),
    "max_abs_corr": round(float(max_abs_corr), 4),
    "max_vif": round(float(max_vif), 2),
    "pool_size": len(vif_pool),
    "pass_fail": "PASS" if all_pass else "FAIL",
}
os.makedirs(os.path.join(REPORTS, "run_summaries"), exist_ok=True)
with open(os.path.join(REPORTS, "run_summaries", "s03_L4.json"), "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"\nRun summary: {REPORTS}/run_summaries/s03_L4.json")

for desc, ok in checks:
    assert ok, f"ACCEPTANCE FAILED: {desc}"

print(f"\n\u2705 All acceptance criteria passed.  Pool size = {len(vif_pool)}")

In [0]:
# ── Good push: strip outputs from src/ notebooks, reimport to correct paths ──
import base64, json as _json
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ExportFormat, ImportFormat

w = WorkspaceClient()
REPO = "/Users/zheng.zhu@rakbank.ae/credit_card_application_playground"

# All src notebooks that need output stripping
# key = target path in repo, value = current workspace path
notebooks_to_strip = {
    # New notebooks created at src/src/ (double-nested) — reimport to src/
    "src/s03_L2": f"{REPO}/src/src/s03_L2",
    "src/s03_L3": f"{REPO}/src/src/s03_L3",
    "src/s03_L4": f"{REPO}/src/src/s03_L4",
    # Existing notebooks already at correct paths
    "src/s01b_data_prep": f"{REPO}/src/s01b_data_prep",
    "src/s02_split": f"{REPO}/src/s02_split",
    "src/s03_L0": f"{REPO}/src/s03_L0",
    "src/s03_L1": f"{REPO}/src/s03_L1",
}

for target_name, src_path in notebooks_to_strip.items():
    print(f"Stripping {target_name}...")
    try:
        resp = w.workspace.export(path=src_path, format=ExportFormat.JUPYTER)
    except Exception as e:
        print(f"  SKIP: {e}")
        continue
    nb_bytes = base64.b64decode(resp.content)
    nb = _json.loads(nb_bytes)

    for cell in nb.get('cells', []):
        cell['outputs'] = []
        cell['execution_count'] = None

    clean_bytes = _json.dumps(nb, indent=1).encode('utf-8')
    clean_b64 = base64.b64encode(clean_bytes).decode('utf-8')

    dest = f"{REPO}/{target_name}"
    w.workspace.import_(
        path=dest,
        content=clean_b64,
        format=ImportFormat.JUPYTER,
        overwrite=True
    )
    print(f"  {len(nb_bytes):,} -> {len(clean_bytes):,} bytes -> {target_name}")

print("\nAll notebooks stripped and imported to correct paths.")